# Text classification

## Adding a hardware accelerator

Please go to the menu and add a GPU as follows:

`Edit > Notebook Settings > Hardware accelerator > (GPU)`

Run the following cell to confirm that the GPU is detected.

In [ ]:
import torch

# Confirm that the GPU is detected

assert torch.cuda.is_available()

# Get the GPU device name.
device_name = torch.cuda.get_device_name()
n_gpu = torch.cuda.device_count()
print(f"Found device: {device_name}, n_gpu: {n_gpu}")
device = torch.device("cuda")

Found device: Tesla T4, n_gpu: 1


In [ ]:
import torch
device = torch.device("cpu")

## Installing Hugging Face's Transformers library
We will use Hugging Face's Transformers (https://github.com/huggingface/transformers), an open-source library that provides general-purpose architectures for natural language understanding and generation with a collection of various pretrained models made by the NLP community. This library will allow us to easily use pretrained models like `BERT` and perform experiments on top of them. We can use these models to solve downstream target tasks, such as text classification, question answering, and sequence labeling.

Run the following cell to install Hugging Face's Transformers library and download a sample data file called tweets.csv that contains tweets about airlines along with a negative, neutral, or positive sentiment rating. Note that you will be asked to link with your Google Drive account to download some of these files. If you're concerned about security risks (there have not been any issues in previous semesters), feel free to make a new Google account and use it for this homework!

In [ ]:
!pip install transformers
!pip install -U -q PyDrive
!pip install nlpaug

import os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from transformers import *
import zipfile
import random
import nlpaug.augmenter.word as naw


# Set the following to avoid warning message
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Applying augmentation to all the selected texts
# Since this may take a while, we shall show progressbar using tqdm
from tqdm.auto import tqdm
tqdm.pandas()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
# Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
print('success!')



# Download helper functions file
helper_file = drive.CreateFile({'id': '16HW-z9Y1tM3gZ_vFpJAuwUDohz91Aac-'})
helper_file.GetContentFile('helpers.py')
print('helper file downloaded! (helpers.py)')


# Download sample file of tweets
data_file = drive.CreateFile({'id': '1iFkLKs6WNoLbttPmpI-3stisQ8QH_iYC'})
data_file.GetContentFile('student_feedback.csv')
print('students feedback comments downloaded! (student_feedback.csv)')


# # Download sample file of tweets
# data_file = drive.CreateFile({'id': '1QcoAmjOYRtsMX7njjQTYooIbJHPc6Ese'})
# data_file.GetContentFile('tweets.csv')
# print('sample tweets downloaded! (tweets.csv)')

success!
helper file downloaded! (helpers.py)
students feedback comments downloaded! (student_feedback.csv)


The cell below imports some helper functions we wrote to demonstrate the task on the sample tweet dataset.

# Part 1: Data Prep and Model Specifications

Upload your data using the file explorer to the left. We have provided a function below to tokenize and format your data as BERT requires. Make sure that your csv file, titled final_data.csv, has one column "text" and another column "labels" containing integers.

If you run the cell below without modifications, it will run on the tweets.csv example data we have provided. It imports some helper functions we wrote to demonstrate the task on the sample tweet dataset. You should first run all of the following cells with tweets.csv just to see how everything works. Then, once you understand the whole preprocessing / fine-tuning process, change the csv in the below cell to your final_data.csv file, add any extra preprocessing code you wish, and then run the cells again on your own data.

In [ ]:
from helpers import tokenize_and_format, flat_accuracy
import pandas as pd
from sklearn.model_selection import train_test_split

# df = pd.read_csv('final_data.csv')
# df = pd.read_csv('tweets.csv')
df = pd.read_csv('student_feedback.csv',encoding='latin-1')
df = df.dropna(subset=['comment'])
df = df.sample(frac=1).reset_index(drop=True)
df.loc[df['label'] == 3, 'label'] = 0

ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)

In [ ]:
df.groupby('label').count()

## Augmentation in training set

We use backtranslation augmentation. For each negative comment, we augmented it three times; from English to German to English, from English to Japaness to English, from English to Spanish to English.


In [ ]:
# Splitting
X_train, X_test, y_train, y_test = train_test_split(df[['comment']], df['label'], test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.175, random_state=42)

In [ ]:
X_train_neg = X_train[y_train == 2]
# X_train_nt = X_train[y_train == 0]

In [ ]:
y_train_neg = y_train[y_train == 2]
# y_train_nt = y_train[y_train == 0]

In [ ]:
train__neg_zh = pd.DataFrame(columns=['comment'])
# train__neg_de = pd.DataFrame(columns=['comment'])
# train__neg_es = pd.DataFrame(columns=['comment'])
# train__nt_zh = pd.DataFrame(columns=['comment'])
# train__nt_de = pd.DataFrame(columns=['comment'])

In [ ]:

# Selecting the back translation augmentation method
# 4) Back translation augmentation
# back_trans_aug = naw.BackTranslationAug(device="cuda")  # If using GPUs
augmenter = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-zh', to_model_name='Helsinki-NLP/opus-mt-zh-en', device=device)

train__neg_zh['comment'] = X_train_neg.progress_apply(lambda row : augmenter.augment(row["comment"])[0], axis=1)

In [ ]:
print(train__neg_zh['comment'])


In [ ]:
# augmenter = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-de', to_model_name='Helsinki-NLP/opus-mt-de-en')

# train__neg_de['comment'] = X_train_neg.progress_apply(lambda row : augmenter.augment(row["comment"])[0], axis=1)

In [ ]:
# augmenter = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-es', to_model_name='Helsinki-NLP/opus-mt-es-en')

# train__neg_es['comment'] = X_train_neg.progress_apply(lambda row : augmenter.augment(row["comment"])[0], axis=1)

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-en-es/snapshots/5bc4493d463cf000c1f0b50f8d56886a392ed4ab/config.json
Model config MarianConfig {
  "_name_or_path": "Helsinki-NLP/opus-mt-en-es",
  "activation_dropout": 0.0,
  "activation_function": "swish",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "MarianMTModel"
  ],
  "attention_dropout": 0.0,
  "bad_words_ids": [
    [
      65000
    ]
  ],
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 512,
  "decoder_attention_heads": 8,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 65000,
  "decoder_vocab_size": 65001,
  "dropout": 0.1,
  "encoder_attention_heads": 8,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 0,
  "extra_pos_embeddings": 65001,
  "force_bos_token_to_be_generated": false

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-en-es/snapshots/5bc4493d463cf000c1f0b50f8d56886a392ed4ab/pytorch_model.bin
Attempting to create safetensors variant
Generate config GenerationConfig {
  "bad_words_ids": [
    [
      65000
    ]
  ],
  "bos_token_id": 0,
  "decoder_start_token_id": 65000,
  "eos_token_id": 0,
  "forced_eos_token_id": 0,
  "max_length": 512,
  "num_beams": 4,
  "pad_token_id": 65000
}

Attempting to convert .bin model on the fly to safetensors.
All model checkpoint weights were used when initializing MarianMTModel.

All the weights of MarianMTModel were initialized from the model checkpoint at Helsinki-NLP/opus-mt-en-es.
If your task is similar to the task the model of the checkpoint was trained on, you can already use MarianMTModel for predictions without further training.


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

loading file source.spm from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-es-en/snapshots/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/source.spm
loading file target.spm from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-es-en/snapshots/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/target.spm
loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-es-en/snapshots/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/vocab.json
loading file target_vocab.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-es-en/snapshots/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/tokenizer_config.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer.json from cache at None
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Helsinki-NLP--opus-mt-es-

  0%|          | 0/49 [00:00<?, ?it/s]

In [ ]:
# augmenter = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-zh', to_model_name='Helsinki-NLP/opus-mt-zh-en')

# train__nt_zh['comment'] = X_train_nt.progress_apply(lambda row : augmenter.augment(row["comment"])[0], axis=1)

  0%|          | 0/55 [00:00<?, ?it/s]

In [ ]:
# augmenter = naw.BackTranslationAug(from_model_name='Helsinki-NLP/opus-mt-en-de', to_model_name='Helsinki-NLP/opus-mt-de-en')

# train__nt_de['comment'] = X_train_nt.progress_apply(lambda row : augmenter.augment(row["comment"])[0], axis=1)

  0%|          | 0/55 [00:00<?, ?it/s]

In [ ]:
# X_train_aug = pd.concat([X_train, train__neg_zh, train__neg_de, train__neg_es, train__nt_zh, train__nt_de], axis=0, ignore_index=True)
X_train_aug = pd.concat([X_train, train__neg_zh], axis=0, ignore_index=True)

In [ ]:
# y_train_aug = pd.concat([y_train, y_train_neg, y_train_neg, y_train_neg, y_train_nt, y_train_nt], axis=0, ignore_index=True)
y_train_aug = pd.concat([y_train, y_train_neg], axis=0, ignore_index=True)

In [ ]:
y_train_aug.groupby(y_train_aug).count()

,label
label,
0,109
1,101
2,108


In [ ]:
train_aug = pd.concat([X_train_aug, y_train_aug], axis=1)
# Shuffle the DataFrame rows
train_aug_shuffled = train_aug.sample(frac=1, random_state=42)

# Extract the shuffled Series
X_train = train_aug_shuffled['comment']
y_train = train_aug_shuffled['label']

In [ ]:
num_train = X_train.shape[0]
num_val = X_val.shape[0]
num_test = X_test.shape[0]
total = num_train + num_val + num_test

In [ ]:
print("Train set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Train set: (318,) (318,)
Validation set: (56, 1) (56,)
Test set: (57, 1) (57,)


In [ ]:
X = pd.concat([X_train, X_val, X_test], axis=0)
y = pd.concat([y_train, y_val, y_test], axis=0)
texts = X.comment.values
labels = y.values

In [ ]:
y.groupby(y).count()

,label
label,
0,152
1,147
2,132


In [ ]:
# texts = df.comment.values
# # texts = df.text.values
# # df.label[df.label == 'O'] = 1
# # df.label[df.label == 'N'] = 0
# df.label[df.label == 3] = 0
# df.label = df.label.astype('int64')
# labels = df.label.values

In [ ]:
texts.shape

(431,)

In [ ]:
labels.shape

(431,)

## Create train/test/validation splits

Here we split your dataset into 3 parts: a training set, a validation set, and a testing set. Each item in your dataset will be a 3-tuple containing an input_id tensor, an attention_mask tensor, and a label tensor.



In [ ]:
### tokenize_and_format() is a helper function provided in helpers.py ###
input_ids, attention_masks = tokenize_and_format(texts)

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

# Print sentence 0, now as a list of IDs.
print('Original: ', texts[0])
print('Token IDs:', input_ids[0])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

loading file vocab.txt from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/vocab.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer_config.json
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer.json


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.46.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



Original:  Virtual Classes and questions that was given by Professor.
Token IDs: tensor([ 101, 7484, 4280, 1998, 3980, 2008, 2001, 2445, 2011, 2934, 1012,  102,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])


In [ ]:
train_set = [(input_ids[i], attention_masks[i], labels[i]) for i in range(num_train)]
train_text = [texts[i] for i in range(num_train)]

val_set = [(input_ids[i], attention_masks[i], labels[i]) for i in range(num_train, num_val+ num_train)]
test_set = [(input_ids[i], attention_masks[i], labels[i]) for i in range(num_val + num_train, total)]

val_text = [texts[i] for i in range(num_train, num_val+num_train)]
test_text = [texts[i] for i in range(num_val + num_train, total)]

Here we choose the model we want to finetune from https://huggingface.co/transformers/pretrained_models.html. Because the task requires us to label sentences, we wil be using BertForSequenceClassification below. You may see a warning that states that `some weights of the model checkpoint at [model name] were not used when initializing. . .` This warning is expected and means that you should fine-tune your pre-trained model before using it on your downstream task. See [here](https://github.com/huggingface/transformers/issues/5421#issuecomment-652582854) for more info.

In [ ]:
from transformers import BertForSequenceClassification, AdamW, BertConfig

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = 3, # The number of output labels.
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

# Tell pytorch to run this model on the GPU.
model.cuda()


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.46.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

# ACTION REQUIRED #

Define your fine-tuning hyperparameters in the cell below (we have randomly picked some values to start with). We want you to experiment with different configurations to find the one that works best (i.e., highest accuracy) on your validation set. Feel free to also change pretrained models to others available in the HuggingFace library (you'll have to modify the cell above to do this). You might find papers on BERT fine-tuning stability (e.g., [Mosbach et al., ICLR 2021](https://openreview.net/pdf?id=nzpLWnVAyah)) to be of interest.

In [ ]:
batch_size = 32
optimizer = AdamW(model.parameters(),
                  lr = 5e-6, # args.learning_rate - default is 5e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8
                )
epochs = 30


# Fine-tune your model
Here we provide code for fine-tuning your model, monitoring the loss, and checking your validation accuracy. Rerun both of the below cells when you change your hyperparameters above.

In [ ]:
import numpy as np
# function to get validation accuracy
def get_validation_performance(val_set):
    # Put the model in evaluation mode
    model.eval()

    # Tracking variables
    total_eval_accuracy = 0
    total_eval_loss = 0

    num_batches = int(len(val_set)/batch_size) + 1

    total_correct = 0

    for i in range(num_batches):

      end_index = min(batch_size * (i+1), len(val_set))

      batch = val_set[i*batch_size:end_index]

      if len(batch) == 0: continue

      input_id_tensors = torch.stack([data[0] for data in batch])
      input_mask_tensors = torch.stack([data[1] for data in batch])
      label_tensors = torch.stack([data[2] for data in batch])

      # Move tensors to the GPU
      b_input_ids = input_id_tensors.to(device)
      b_input_mask = input_mask_tensors.to(device)
      b_labels = label_tensors.to(device)

      # Tell pytorch not to bother with constructing the compute graph during
      # the forward pass, since this is only needed for backprop (training).
      with torch.no_grad():

        # Forward pass, calculate logit predictions.
        outputs = model(b_input_ids,
                                token_type_ids=None,
                                attention_mask=b_input_mask,
                                labels=b_labels)
        loss = outputs.loss
        logits = outputs.logits

        # Accumulate the validation loss.
        total_eval_loss += loss.item()

        # Move logits and labels to CPU
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # Calculate the number of correctly labeled examples in batch
        pred_flat = np.argmax(logits, axis=1).flatten()
        labels_flat = label_ids.flatten()
        num_correct = np.sum(pred_flat == labels_flat)
        total_correct += num_correct

    # Report the final accuracy for this validation run.
    avg_val_accuracy = total_correct / len(val_set)
    return avg_val_accuracy



In [ ]:
import random

# training loop

# For each epoch...
for epoch_i in range(0, epochs):
    # Perform one full pass over the training set.

    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    # Reset the total loss for this epoch.
    total_train_loss = 0

    # Put the model into training mode.
    model.train()

    # For each batch of training data...
    num_batches = int(len(train_set)/batch_size) + 1

    for i in range(num_batches):
      end_index = min(batch_size * (i+1), len(train_set))

      batch = train_set[i*batch_size:end_index]

      if len(batch) == 0: continue

      input_id_tensors = torch.stack([data[0] for data in batch])
      input_mask_tensors = torch.stack([data[1] for data in batch])
      label_tensors = torch.stack([data[2] for data in batch])

      # Move tensors to the GPU
      b_input_ids = input_id_tensors.to(device)
      b_input_mask = input_mask_tensors.to(device)
      b_labels = label_tensors.to(device)

      # Clear the previously calculated gradient
      model.zero_grad()

      # Perform a forward pass (evaluate the model on this training batch).
      outputs = model(b_input_ids,
                            token_type_ids=None,
                            attention_mask=b_input_mask,
                            labels=b_labels)
      loss = outputs.loss
      logits = outputs.logits

      total_train_loss += loss.item()

      # Perform a backward pass to calculate the gradients.
      loss.backward()

      # Update parameters and take a step using the computed gradient.
      optimizer.step()

    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set. Implement this function in the cell above.
    print(f"Total loss: {total_train_loss}")
    val_acc = get_validation_performance(val_set)
    print(f"Validation accuracy: {val_acc}")

print("")
print("Training complete!")



======== Epoch 1 / 30 ========
Training...
Total loss: 11.353326201438904
Validation accuracy: 0.23214285714285715

======== Epoch 2 / 30 ========
Training...
Total loss: 10.762079000473022
Validation accuracy: 0.32142857142857145

======== Epoch 3 / 30 ========
Training...
Total loss: 10.277157187461853
Validation accuracy: 0.48214285714285715

======== Epoch 4 / 30 ========
Training...
Total loss: 9.750039100646973
Validation accuracy: 0.5892857142857143

======== Epoch 5 / 30 ========
Training...
Total loss: 9.398296058177948
Validation accuracy: 0.6607142857142857

======== Epoch 6 / 30 ========
Training...
Total loss: 8.795197427272797
Validation accuracy: 0.6964285714285714

======== Epoch 7 / 30 ========
Training...
Total loss: 8.225375652313232
Validation accuracy: 0.75

======== Epoch 8 / 30 ========
Training...
Total loss: 7.723346531391144
Validation accuracy: 0.75

======== Epoch 9 / 30 ========
Training...
Total loss: 7.23771071434021
Validation accuracy: 0.76785714285714

# Evaluate your model on the test set
After you're satisfied with your hyperparameters (i.e., you're unable to achieve higher validation accuracy by modifying them further), it's time to evaluate your model on the test set! Run the below cell to compute test set accuracy.


In [ ]:
get_validation_performance(test_set)

0.8947368421052632

In [ ]:
def error_analysis(tensor_set, text_set):
    model.eval()
    batch = tensor_set

    input_id_tensors = torch.stack([data[0] for data in batch])
    input_mask_tensors = torch.stack([data[1] for data in batch])
    label_tensors = torch.stack([data[2] for data in batch])

    # Move tensors to the GPU
    b_input_ids = input_id_tensors.to(device)
    b_input_mask = input_mask_tensors.to(device)
    b_labels = label_tensors.to(device)

    # Tell pytorch not to bother with constructing the compute graph during
    # the forward pass, since this is only needed for backprop (training).
    with torch.no_grad():

        # Forward pass, calculate logit predictions.
        outputs = model(b_input_ids,
                                token_type_ids=None,
                                attention_mask=b_input_mask,
                                labels=b_labels)

        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # Calculate the number of correctly labeled examples in batch
        pred_flat = np.argmax(logits, axis=1).flatten()
        labels_flat = label_ids.flatten()

        for i in range(labels_flat.size):
          if pred_flat[i] != labels_flat[i]:
            print("Incorrect -- " + text_set[i])
          else:
            print("Correct -- " + text_set[i])

In [ ]:

error_analysis(val_set, val_text)

Correct -- I liked that this class was very discussion–based. I also liked how the professor went through practice problems for the energy
portion of the course.
Incorrect -- nothing
Incorrect -- How the activities aided the content we were learning
Correct -- allow for more in class interaction and open yourself up for more questions during lectures
Correct -- It was good
Correct -- Overall it was a good course, and that’s coming from someone who has absolutely no interest in LEED.
Correct -- The way instructor teaching us this course makes more interest to gain knowledge and also to participate in activities in class and
 many more. So, I like every aspects of this course.
Incorrect -- Be prepared for a lot of homework, which is a repeat of the materials class we took sophomore year. Also, the lab reports are very 
unclear, and require a lot of time and effort.
Incorrect -- she is friendly – fast response – but sometimes so tough
Correct -- Take this subject, it would help in better 

In [ ]:
error_analysis(test_set, test_text)

Incorrect -- Nothing
Correct -- doing more inclass problems
Correct -- Mailing the concrete ingredients was very nice.
Correct -- Always willing to help students.
Incorrect -- Be a little more clear when making plans
Incorrect -- To be honest I don't have any suggestions
Correct -- Interesting
Incorrect -- You will not have a break from this class until the final exam is given.
Incorrect -- The homework although helpful, was a large load at times, especially when compared to other the fundamentals classes. An 
example based class is nice but, the lectures were sometimes to brief and to careless. I felt the lectures would confused me 
more, while the examples would provide SOME clarity. Much learning was spent by me, teaching myself and seeing the trends 
myself. A good class has balance of all things, and just remember your students cant see things or do things as quick as you!
Correct -- I enjoyed my first semester as an engineering student. I believe achieving a high grade in this co

In [ ]:
error_analysis(train_set, train_text)

Correct -- Take really good notes.
Correct -- All of the assignments and modules we had to do were clearly organized on Canvas. My instructor was very approachable and
Incorrect -- My experience was overall difficultly challenging me in this course
Incorrect -- Be more careful while walking down stairs
Correct -- She thought us Many examples to understand the subject.
Correct -- Dr. Nossoni has a high energy that I absolutely love during college. I never feel tired whenever I am in her class and is always 
helpful with questions.
Incorrect -- keep up
Correct -- I enjoyed the lectures and how my professor was able to make each class relatable to the real world.
Incorrect -- The course was taught very well. The only thing that made this semester more challenging than usual was the unexpected transition 
to online courses. The online exams were extremely difficult, and I imagine in–class exams would be easier to complete in the given 
time frame. If this class is online again in the futur